# AI Inference Worker trên Google Colab (T4 GPU)

Notebook này khởi chạy `BE/worker/service.py` trên Google Colab để kết nối tới Backend Azure VM (`http://70.153.17.0`).

### Chuẩn bị trước khi bấm **Run all**
1. Chọn GPU: **Runtime → Change runtime type → T4 GPU**.
2. Upload tệp weights `best.pt` lên Google Drive (mặc định `/content/drive/MyDrive/PBL6/best.pt`).
3. Lưu `WORKER_TOKEN` vào **Colab Secrets** (không dán token trực tiếp vào notebook):
   - Bấm biểu tượng **chìa khóa (Secrets)** ở thanh bên trái Colab.
   - Chọn **Add new secret**: Name = `WORKER_TOKEN`, Value = token từ `/opt/bts/.env` trên Azure VM.
   - Bật công tắc **Notebook access** cho secret vừa tạo.
4. Bấm **Runtime → Run all**. *(Nếu chưa cấu hình Colab Secret, Cell cuối sẽ hiện ô nhập ẩn bằng `getpass`.)*


In [ ]:
#@title 1. Cấu hình kết nối & Mount Google Drive
import shutil
import subprocess
from pathlib import Path

WORKER_API_BASE_URL = "http://70.153.17.0"  #@param {type:"string"}
WORKER_ID = "colab-t4-worker-1"  #@param {type:"string"}
WORKER_DEVICE = "auto"  #@param ["auto", "cuda", "cpu"]
WORKER_POLL_INTERVAL_SECONDS = 5  #@param {type:"integer"}
REPO_URL = "https://github.com/hwolp/Brain-Tumor-Segmentation.git"  #@param {type:"string"}
REPO_BRANCH = "master"  #@param {type:"string"}
MODEL_CHECKPOINT_PATH = "/content/drive/MyDrive/PBL6/best.pt"  #@param {type:"string"}

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Không chạy trong Google Colab; bỏ qua bước mount Google Drive.")

if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
else:
    print("Không tìm thấy nvidia-smi (đang chạy CPU).")

checkpoint_file = Path(MODEL_CHECKPOINT_PATH)
if not checkpoint_file.is_file():
    raise FileNotFoundError(
        f"Không tìm thấy checkpoint tại '{checkpoint_file}'. "
        "Hãy kiểm tra lại đường dẫn file best.pt trên Google Drive."
    )
print(f"Đã tìm thấy checkpoint: {checkpoint_file} ({checkpoint_file.stat().st_size / (1024 * 1024):.1f} MiB)")


In [ ]:
#@title 2. Đồng bộ mã nguồn, cài thư viện & xác minh SHA-256 checkpoint
import hashlib
import importlib.util
import json
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path("/content/bts")
if not (REPO_DIR / ".git").is_dir():
    subprocess.run(["rm", "-rf", str(REPO_DIR)], check=True)
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{REPO_BRANCH}"],
        check=True,
    )

packages = [
    "httpx==0.28.1",
    "nibabel==5.5.0",
    "gudhi==3.13.0",
    "psutil==7.0.0",
    "scikit-image==0.26.0",
]
if importlib.util.find_spec("torch") is None:
    packages.insert(0, "torch==2.8.0")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)

import gudhi
import httpx
import nibabel
import psutil
import scipy
import skimage
import torch

print(
    f"torch={torch.__version__}, cuda_available={torch.cuda.is_available()}, "
    f"gudhi={gudhi.__version__}, nibabel={nibabel.__version__}, httpx={httpx.__version__}, "
    f"skimage={skimage.__version__}, scipy={scipy.__version__}, psutil={psutil.__version__}"
)
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

MANIFEST_PATH = REPO_DIR / "BE" / "worker" / "manifest.json"
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
expected_sha = manifest["checkpoint_sha256"].lower()

digest = hashlib.sha256()
with Path(MODEL_CHECKPOINT_PATH).open("rb") as fh:
    while chunk := fh.read(1024 * 1024):
        digest.update(chunk)
actual_sha = digest.hexdigest().lower()

if actual_sha != expected_sha:
    raise ValueError(
        f"Checkpoint SHA-256 mismatch! expected={expected_sha}, actual={actual_sha}"
    )
print(f"Xác minh SHA-256 checkpoint thành công: {actual_sha}")


In [ ]:
#@title 3. Kiểm tra kết nối tới Azure VM Backend
import httpx

health_url = f"{WORKER_API_BASE_URL.rstrip('/')}/api/v1/health/ready"
resp = httpx.get(health_url, timeout=15.0)
print(f"GET {health_url} -> HTTP {resp.status_code}")
print(resp.text)
resp.raise_for_status()


In [ ]:
#@title 4. Nạp WORKER_TOKEN an toàn & chạy AI Worker
import getpass
import os
import subprocess
import sys


def load_worker_token() -> str:
    try:
        from google.colab import userdata
        secret_val = userdata.get("WORKER_TOKEN")
        if secret_val and secret_val.strip():
            print("Đã nạp WORKER_TOKEN từ Colab Secrets.")
            return secret_val.strip()
    except Exception:
        pass

    env_val = os.environ.get("WORKER_TOKEN", "").strip()
    if env_val:
        print("Đã nạp WORKER_TOKEN từ biến môi trường.")
        return env_val

    return getpass.getpass("Nhập WORKER_TOKEN từ /opt/bts/.env: ").strip()


worker_token = load_worker_token()
if not worker_token:
    raise ValueError("WORKER_TOKEN không được để trống.")

env = os.environ.copy()
env.update(
    {
        "WORKER_API_BASE_URL": WORKER_API_BASE_URL.rstrip("/"),
        "WORKER_TOKEN": worker_token,
        "WORKER_ID": WORKER_ID,
        "WORKER_DEVICE": WORKER_DEVICE,
        "WORKER_POLL_INTERVAL_SECONDS": str(WORKER_POLL_INTERVAL_SECONDS),
        "MODEL_MANIFEST_PATH": "/content/bts/BE/worker/manifest.json",
        "MODEL_CHECKPOINT_PATH": MODEL_CHECKPOINT_PATH,
        "LOG_LEVEL": "INFO",
    }
)

print(
    f"Đang khởi động worker_id={WORKER_ID} (device={WORKER_DEVICE}) "
    f"kết nối tới {env['WORKER_API_BASE_URL']}..."
)
proc = subprocess.Popen(
    [sys.executable, "-m", "worker.service"],
    cwd="/content/bts/BE",
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

try:
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="", flush=True)
    exit_code = proc.wait()
    if exit_code != 0:
        raise RuntimeError(f"Worker thoát với mã lỗi {exit_code}")
except KeyboardInterrupt:
    print("\nĐang dừng AI Worker...")
    proc.terminate()
    try:
        proc.wait(timeout=10)
    except subprocess.TimeoutExpired:
        proc.kill()
        proc.wait()
    print("AI Worker đã dừng.")
